# Region-level ablation: is the ground truth clean at OBJECT granularity?

Everything so far measured importance **per 9x9 patch**. But a patch is not an object — one dog
spans six patches, one patch may be half dog and half counter. That mismatch predicts three of our
confusing results:

| symptom we saw | what object-granularity predicts |
|---|---|
| 74% of patches had near-zero drop | cover one patch of the dog, the model sees the other five and answers fine |
| `gaze-sim` scored at random | comparing two *squares* for visual similarity is not "is this object related to that object" — a price tag does not look like a product |
| FRM memorised (87.5% train, 18.7% held-out) | predicting over 63 patches from 80 examples is a huge output space; over ~8 regions it is a small one |

**The test.** Group the 81 tokens into ~8 object-ish regions, then ablate **whole regions** instead
of single patches. If the granularity diagnosis is right, the drops should become **sharp and
non-redundant** where per-patch drops were mostly noise.

Also asks the far-context question at the level it actually makes sense: **is the region the answer
needed the same region you were looking at, or a different one?**

Cost: ~9 forwards per example instead of 81 — about **10x cheaper** than the patch version.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy scikit-learn
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, time, gc
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import visual_selection as VS

N100  = "/content/drive/MyDrive/wearvqa_n100.pt"
EMB   = "/content/drive/MyDrive/wearvqa_n100_emb.pt"
SINKF = "/content/drive/MyDrive/sink_mask_smolvlm2_n12.pt"
OUT   = "/content/drive/MyDrive/wearvqa_regions.pt"
MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

K_REG     = 8        # regions per image
SPATIAL_W = 0.8      # how much spatial contiguity to force into the clustering
PCA_DIM   = 16

for p in (N100, EMB):
    assert os.path.exists(p), f"{p} missing - run colab_n100_scaleup then colab_train_frm"
data  = [d for d in torch.load(N100, weights_only=False) if "gp" in d]
embs  = torch.load(EMB, weights_only=False)
sinks = torch.load(SINKF, weights_only=False)["sink_mask"].bool()

L_v = data[0]["drops"].numel(); G = int(round(math.sqrt(L_v))); N = len(data)
print(f"{N} examples | L_v={L_v} ({G}x{G}) | K={K_REG} regions")

## 2. Group patches into regions

KMeans over each image's token embeddings (PCA'd to 16 dims and L2-normalised) with the grid
coordinates appended, so regions come out **spatially contiguous** rather than scattered.

In [ ]:
rows_, cols_ = np.divmod(np.arange(L_v), G)
XY = np.stack([rows_, cols_], 1) / G

def regions_for(E, k=K_REG):
    Z = PCA(PCA_DIM, random_state=0).fit_transform(
        F.normalize(E.float(), dim=-1).numpy())
    Z = Z / np.clip(np.linalg.norm(Z, axis=1, keepdims=True), 1e-6, None)
    feat = np.concatenate([Z, XY * SPATIAL_W], 1)
    return torch.tensor(KMeans(k, n_init=10, random_state=0).fit_predict(feat))

labels = [regions_for(e) for e in embs]
sizes = np.array([[int((l == r).sum()) for r in range(K_REG)] for l in labels])
print(f"region sizes: mean {sizes.mean():.1f} patches, "
      f"min {sizes.min()}, max {sizes.max()}")

# contiguity check: how many 4-connected components does each region break into?
def n_components(mask2d):
    seen = np.zeros_like(mask2d, dtype=bool); n = 0
    for i in range(G):
        for j in range(G):
            if mask2d[i, j] and not seen[i, j]:
                n += 1; stack = [(i, j)]; seen[i, j] = True
                while stack:
                    a, b = stack.pop()
                    for da, db in ((1,0),(-1,0),(0,1),(0,-1)):
                        p, q = a+da, b+db
                        if 0 <= p < G and 0 <= q < G and mask2d[p, q] and not seen[p, q]:
                            seen[p, q] = True; stack.append((p, q))
    return n

comps = [n_components((l.reshape(G, G) == r).numpy())
         for l in labels[:20] for r in range(K_REG)]
print(f"connected components per region: mean {np.mean(comps):.2f} "
      f"(1.0 = perfectly contiguous)")

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(15, 7))
for a, i in zip(ax.ravel(), range(8)):
    img = S.load_image(data[i]["img_path"]); W, H = img.size
    from PIL import Image as _I
    seg = _I.fromarray((labels[i].reshape(G, G).numpy() * 255 // K_REG).astype("uint8")).resize((W, H), _I.NEAREST)
    a.imshow(img); a.imshow(np.array(seg), cmap="tab10", alpha=0.55)
    gp = data[i]["gp"]
    a.scatter([(gp % G + .5) / G * W], [(gp // G + .5) / G * H],
              marker="x", s=140, c="lime", linewidths=3)
    a.set_title(f"{data[i]['type'][:22]}", fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()
print("green x = gaze. Regions should look like objects/surfaces, not confetti.")

## 3. Ablate whole regions

Mask every token in a region at once — but **never the sink tokens**, so a region that happens to
contain a corner does not get credit for destabilising the model.

In [ ]:
model, processor, device = S._load_smolvlm(MODEL_ID)
tokenizer = processor.tokenizer

def build_inputs(image, question, answer):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    return full, int(only["input_ids"].shape[1])

@torch.no_grad()
def answer_logprob(inp, n_prompt, attention_mask=None):
    kw = dict(inp)
    if attention_mask is not None:
        kw["attention_mask"] = attention_mask
    logits = model(**kw).logits[0].float()
    lp = torch.log_softmax(logits[:-1], dim=-1)
    return float(lp.gather(-1, inp["input_ids"][0, 1:, None]).squeeze(-1)[n_prompt-1:].sum())

if os.path.exists(OUT):
    regs = torch.load(OUT, weights_only=False)
    print(f"loaded {len(regs)} cached region results")
else:
    regs, t0 = [], time.time()
    for i, d in enumerate(data):
        inp, n_prompt = build_inputs(S.load_image(d["img_path"]), d["question"], d["answer"])
        ids = inp["input_ids"][0].cpu()
        img_pos = torch.nonzero(ids == S._find_image_token_id(model, processor)).squeeze(-1)
        base = answer_logprob(inp, n_prompt)
        lab = labels[i]
        rd = torch.zeros(K_REG)
        for r in range(K_REG):
            sel = ((lab == r) & (~sinks[:L_v])).nonzero().squeeze(-1)   # never mask sinks
            if len(sel) == 0:
                continue
            am = inp["attention_mask"].clone(); am[0, img_pos[sel]] = 0
            rd[r] = base - answer_logprob(inp, n_prompt, am)
        regs.append(dict(idx=i, base=base, region_drops=rd, labels=lab,
                         gaze_region=int(lab[d["gp"]])))
        if (i + 1) % 25 == 0:
            print(f"  {i+1}/{N}  ({(time.time()-t0)/60:.1f} min)")
    torch.save(regs, OUT)
    print(f"done in {(time.time()-t0)/60:.1f} min -> {OUT}")

## 4. The diagnostic — did the ground truth get sharper?

Per-patch ablation gave 74% near-zero drops and a top unit holding ~21% of the total. If objects are
the right unit, region drops should be **larger in magnitude** and **more concentrated**.

In [ ]:
def stats(vals_list, label, near_zero=0.05):
    nz, mx, share, ratio = [], [], [], []
    for v in vals_list:
        v = v.float(); a = v.abs()
        nz.append(float((a < near_zero).float().mean()))
        mx.append(float(v.max()))
        share.append(float(a.max() / a.sum().clamp_min(1e-9)))
        ratio.append(float(a.max() / a.median().clamp_min(1e-9)))
    print(f"{label:<22}{np.mean(nz):>12.0%}{np.mean(mx):>12.3f}"
          f"{np.mean(share):>12.0%}{np.mean(ratio):>12.1f}x")

print(f"{'unit':<22}{'near-zero':>12}{'max drop':>12}{'top share':>12}{'max/median':>12}")
print("-" * 70)
stats([d["drops"] for d in data], "per PATCH (81)")
stats([r["region_drops"] for r in regs], f"per REGION ({K_REG})")

allr = torch.cat([r["region_drops"] for r in regs])
allp = torch.cat([d["drops"] for d in data])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(allp.numpy(), bins=60); ax[0].set_title("per-patch drops"); ax[0].axvline(0, c="k", lw=1)
ax[1].hist(allr.numpy(), bins=40); ax[1].set_title(f"per-region drops (K={K_REG})"); ax[1].axvline(0, c="k", lw=1)
for a in ax:
    a.set_xlabel("drop in answer logprob")
plt.tight_layout(); plt.show()
print("\nIf the region histogram has a real right tail where the patch one had a spike at 0,")
print("the granularity diagnosis is confirmed and every earlier measurement needs redoing.")

## 5. Is the region the answer needed the one you were looking at?

The far-context question, finally asked at the granularity where it makes sense. `FRM`'s whole job
is the case where the answer needed a **different** region from the gaze one.

In [ ]:
same, gaze_rank, top_is_gaze = 0, [], []
far_examples = []
for r, d in zip(regs, data):
    rd, gr = r["region_drops"], r["gaze_region"]
    top = int(rd.argmax())
    same += int(top == gr)
    order = torch.argsort(rd, descending=True).tolist()
    gaze_rank.append(order.index(gr) + 1)
    if top != gr:
        far_examples.append((r["idx"], gr, top, float(rd[top]), float(rd[gr])))

print(f"top-drop region IS the gaze region : {same}/{len(regs)}  ({same/len(regs):.0%})")
print(f"top-drop region is ELSEWHERE       : {len(regs)-same}/{len(regs)}  "
      f"({1-same/len(regs):.0%})   <- FRM's reason to exist")
print(f"\ngaze region's average rank among {K_REG} regions: {np.mean(gaze_rank):.2f} "
      f"(1 = always the most important, {(K_REG+1)/2:.1f} = no better than chance)")

# how much of the total damage sits outside the gaze region?
outside = [float(r["region_drops"].abs().sum() - r["region_drops"][r["gaze_region"]].abs())
           / float(r["region_drops"].abs().sum().clamp_min(1e-9)) for r in regs]
print(f"share of total |drop| OUTSIDE the gaze region: {np.mean(outside):.0%}")

by = defaultdict(list)
for r, d in zip(regs, data):
    by[d["type"]].append(int(int(r["region_drops"].argmax()) != r["gaze_region"]))
print("\nper question type - fraction where the answer needed a DIFFERENT region:")
for t in sorted(by):
    print(f"   {t:<38} {np.mean(by[t]):.0%}")

In [ ]:
# look at a few far-context cases: gaze on one object, the answer needed another
show = far_examples[:6]
fig, ax = plt.subplots(len(show), 2, figsize=(9, 3.3 * len(show)))
if len(show) == 1:
    ax = ax[None, :]
from PIL import Image as _I
for row, (i, gr, top, dtop, dgaze) in enumerate(show):
    d = data[i]; img = S.load_image(d["img_path"]); W, H = img.size
    lab = labels[i]
    ax[row, 0].imshow(img); ax[row, 0].axis("off")
    ax[row, 0].set_title(f"{d['question'][:64]}", fontsize=7)
    heat = torch.zeros(L_v)
    for rr in range(K_REG):
        heat[lab == rr] = regs[i]["region_drops"][rr]
    h = heat.reshape(G, G).numpy(); h = (h - h.min()) / (np.ptp(h) + 1e-9)
    ax[row, 1].imshow(img)
    ax[row, 1].imshow(np.array(_I.fromarray((h*255).astype("uint8")).resize((W, H))),
                      cmap="jet", alpha=0.5)
    ax[row, 1].axis("off")
    ax[row, 1].set_title(f"region damage | gaze region {gr} ({dgaze:.2f})  "
                         f"top region {top} ({dtop:.2f})", fontsize=7)
    gp = d["gp"]
    for a in ax[row]:
        a.scatter([(gp % G + .5)/G*W], [(gp // G + .5)/G*H],
                  marker="x", s=130, c="lime", linewidths=3)
plt.tight_layout(); plt.show()

## 6. Verdict

**Section 4 — did the granularity diagnosis hold?**
* **Region drops much larger and more concentrated than patch drops** -> confirmed. Per-patch
  ablation was measuring redundancy, not importance, and every earlier number should be redone at
  region level (including the teacher bake-off and FRM training).
* **Region drops look like the patch histogram, just fewer** -> granularity was not the problem.
  The redundancy story is wrong and the weak signal is real.

**Section 5 — does FRM have a job?**
* **Top-drop region is elsewhere in a clear majority** -> the answer routinely needs a region you are
  not looking at, at object granularity. That is the FRM premise, stated properly for the first time.
* **Top-drop region is usually the gaze region** -> Stage 2a's fovea already covers what the answer
  needs, and Stage 2b has little left to retrieve.
* **Gaze region's average rank ~ (K+1)/2** -> gaze carries no information about which region matters,
  which would be worse than either.

**If both look good**, the next build is the object-level FRM: gaze region's pooled embedding as the
query, ~8 region embeddings as keys, a small MLP over the interaction, predicting ~8 numbers instead
of 63. That is a small enough learning problem for 100-680 examples, unlike the patch version.